In [19]:
import spacy
from Bio import Entrez
import pandas as pd
import re
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [20]:
# 1. Ask user to input the variant they want to check

#variant = input("Enter variant (e.g., BRCA1, V600E): ")
variant = "BRCA1 c.5266dupC"

# option to filter by date
#use_date= input("Filter by year? ")
date_filter = "2020"

# Build search query with date filter
query = f"{variant} AND {date_filter}[PDAT]"

print(f"Gene: {variant}")
print(f"Date filter: {date_filter}")

Gene: BRCA1 c.5266dupC
Date filter: 2020


In [21]:
#2. Download papers from PubMed 

from Bio import Entrez
from Bio import Medline

Entrez.email = "jawahir_noor@hotmail.com"


# Search
handle = Entrez.esearch(db="pubmed", term=query, retmax=8)
record = Entrez.read(handle)
ids = record["IdList"]
handle.close()

# Simpler approach using BioPython's built-in parser
# Fetch in Medline format
handle = Entrez.efetch(db="pubmed", id=ids, rettype="medline", retmode="text")
records = Medline.parse(handle)

papers_list = []
for record in records:
    papers_list.append({
        "title": record.get("TI", ""),
        "abstract": record.get("AB", ""),
        "pmid": record.get("PMID", "")
    })
handle.close()

print(papers_list)



[{'title': 'Prevalence and Spectrum of BRCA Germline Variants in Central Italian High Risk or Familial Breast/Ovarian Cancer Patients: A Monocentric Study.', 'abstract': 'Hereditary breast and ovarian cancers are mainly linked to variants in BRCA1/2 genes. Recently, data has shown that identification of BRCA variants has an immediate impact not only in cancer prevention but also in targeted therapeutic approaches. This prospective observational study characterized the overall germline BRCA variant and variant of uncertain significance (VUS) frequency and spectrum in individuals affected by breast (BC) or ovarian cancer (OC) and in healthy individuals at risk by sequencing the entire BRCA genes. Of the 363 probands analyzed, 50 (13.8%) were BRCA1/2 mutated, 28 (7.7%) at BRCA1 and 23 (6.3%) at BRCA2 gene. The variant c.5266dupC p.(Gln1756Profs) was the most frequent alteration, representing 21.4% of the BRCA1 variants and 12.0% of all variants identified. The variant c.6313delA p.(Ile210

In [ ]:
#3. Load model (BioNLP13CG)

# BioNLP model (biomedical entities)
nlp_bionlp = spacy.load("en_ner_bionlp13cg_md")


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_ner_bionlp13cg_md' (0.5.1) was trained with spaCy v3.4.1 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [7]:
#4. Extract biomedical entities (BioNLP13CG ) paper by paper
import pandas as pd
import re

results = []


for i, paper in enumerate(papers_list):

    text = paper["abstract"]

    if not text:
        continue

    # safe limit
    text = text[:5000]

    
    doc_bio = nlp_bionlp(text)


    # BioNLP entities
    for ent in doc_bio.ents:
        results.append({
            "paper_id": i,
            "title": paper["title"],
            "text": ent.text,
            "label": ent.label_,
            "model": "BioNLP13CG"
        })

In [22]:
 # 5. Extract mutation variants (Rule-based + regex) 
mutation_patterns = [
    r"c\.\d+(?:_\d+)?[A-Z]>[A-Z]",        # c.5266C>T  (strict: single base change)
    r"c\.\d+(?:_\d+)?dup[A-Z]?",           # c.5266dupC or c.5266dup
    r"c\.\d+(?:_\d+)?del[A-Z]*",           # c.5266del
    r"c\.\d+(?:_\d+)?ins[A-Z]+",           # c.5266insA
    r"p\.[A-Z][a-z]{2}\d+[A-Z][a-z]{2}",  # p.Gln1756Ter  (strict 3-letter AA)
    r"rs\d{4,}",                            # rs ID (min 4 digits to avoid false hits)
]

def extract_variants(text):
        variants = []
        for pattern in mutation_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            variants.extend(matches)
        return list(set(variants))  # Remove duplicates
    
variants_found = extract_variants(text)

variant_results = []  # New list for variant extraction

# Store variant results
for variant in variants_found:
        variant_results.append({
            "paper_id": i,
            "title": paper["title"],
            "pmid": paper.get("pmid", ""),
            "variant": variant
        })

In [23]:
# 5. Convert to tables
df_entities = pd.DataFrame(results)
df_variants = pd.DataFrame(variant_results)

print("  Entities Found  ")
print(df_entities.head())
print(f"\nTotal entities: {len(df_entities)}")

print("\n Variants Found  ")
print(df_variants.head())
print(f"Total variants: {len(df_variants)}")

# Save results
df_entities.to_csv("results/paper_by_paper_entities.csv", index=False)
df_variants.to_csv("results/extracted_variants.csv", index=False)

print("\nResults saved to 'results/' directory")

  Entities Found  
   paper_id                                              title  \
0         0  Prevalence and Spectrum of BRCA Germline Varia...   
1         0  Prevalence and Spectrum of BRCA Germline Varia...   
2         0  Prevalence and Spectrum of BRCA Germline Varia...   
3         0  Prevalence and Spectrum of BRCA Germline Varia...   
4         0  Prevalence and Spectrum of BRCA Germline Varia...   

              text                 label       model  
0           breast                CANCER  BioNLP13CG  
1  ovarian cancers                CANCER  BioNLP13CG  
2          BRCA1/2  GENE_OR_GENE_PRODUCT  BioNLP13CG  
3           cancer                CANCER  BioNLP13CG  
4           breast                CANCER  BioNLP13CG  

Total entities: 101

 Variants Found  
   paper_id                                              title      pmid  \
0         5  Randomised trial of population-based BRCA test...  31507061   
1         5  Randomised trial of population-based BRCA test...

In [27]:
# ============================================================================
# 6. ClinVar normalization helper  —  gene anchor + sig parsing + validation
# ============================================================================
ENTREZ_BASE  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
ENTREZ_EMAIL = "jawahir_noor@hotmail.com"

# Define gene from your variant input
gene = variant.split()[0]
def _extract_position(variant_str):
    """Pull the numeric position from a HGVS string, e.g. c.5266dupC → 5266."""
    m = re.search(r"(\d+)", variant_str)
    return int(m.group(1)) if m else None

def _position_match(extracted, returned, tolerance=50):
    """
    Return True if the ClinVar HGVS position is within `tolerance` of the
    extracted variant position. Rejects completely wrong matches.
    """
    pos_e = _extract_position(extracted)
    pos_r = _extract_position(returned)
    if pos_e is None or pos_r is None:
        return True   # can't check — allow through
    return abs(pos_e - pos_r) <= tolerance

def _parse_significance(doc):
    """
    ClinVar significance can live in several places in the JSON.
    Try all known locations before giving up.
    """
    # Location 1: clinical_significance.description  (most common)
    sig = doc.get("clinical_significance", {})
    if isinstance(sig, dict):
        desc = sig.get("description", "")
        if desc:
            return desc

    # Location 2: germline_classification.description  (newer ClinVar records)
    germ = doc.get("germline_classification", {})
    if isinstance(germ, dict):
        desc = germ.get("description", "")
        if desc:
            return desc

    # Location 3: variation_set entries
    for entry in doc.get("variation_set", []):
        cs = entry.get("clinical_significance", {})
        if isinstance(cs, dict):
            desc = cs.get("description", "")
            if desc:
                return desc

    return "N/A"

def clinvar_normalize(variant_name, gene_name):
    """
    Return (normalized_hgvs, clinical_significance) from ClinVar.
    - Always anchors search to gene_name[gene] — no cross-gene false matches
    - Validates position proximity before accepting result
    - Tries multiple JSON paths for clinical significance
    """
    # Only use gene-anchored query to prevent wrong-gene matches
    queries = [
        f"{gene_name}[gene] AND {variant_name}[variant name]",
        f"{gene_name}[gene] AND {variant_name}",
    ]

    for query in queries:
        try:
            # Search ClinVar
            r = requests.get(
                f"{ENTREZ_BASE}/esearch.fcgi?db=clinvar&term={quote(query)}"
                f"&retmax=3&retmode=json&email={ENTREZ_EMAIL}",
                timeout=15
            )
            time.sleep(0.34)
            ids = r.json().get("esearchresult", {}).get("idlist", [])
            if not ids:
                continue

            # Try each returned ID, pick first with matching position
            for clinvar_id in ids:
                r2 = requests.get(
                    f"{ENTREZ_BASE}/esummary.fcgi?db=clinvar&id={clinvar_id}"
                    f"&retmode=json&email={ENTREZ_EMAIL}",
                    timeout=15
                )
                time.sleep(0.34)
                result = r2.json().get("result", {})
                doc = result.get(str(clinvar_id), {})

                # Parse HGVS from variation_set
                hgvs = "N/A"
                for entry in doc.get("variation_set", []):
                    cdna = entry.get("cdna_change", "")
                    if cdna:
                        hgvs = cdna
                        break
                if hgvs == "N/A":
                    hgvs = doc.get("title", "N/A")

                # FIX: Validate position matches before accepting
                if hgvs != "N/A" and not _position_match(variant_name, hgvs):
                    continue   # wrong variant — try next ID

                significance = _parse_significance(doc)
                return hgvs, significance

        except Exception:
            continue

    return "N/A", "N/A"



In [28]:
# 6. Build tables, normalize, print and save

import os

os.makedirs("results", exist_ok=True)

df_entities = pd.DataFrame(results)

# Normalize and add columns directly into variant_results rows
print("\nVariants extracted and Normalizing via ClinVar...")
for row in variant_results:
    hgvs, sig = clinvar_normalize(row["variant"], gene)
    row["normalized_hgvs"]       = hgvs
    row["clinical_significance"] = sig
    print(f"  {row['variant']} → {hgvs} [{sig}]")

df_variants = pd.DataFrame(variant_results)

#  Print
print("\n=== Entities Found ===")
print(df_entities.head())
print(f"\nTotal entities: {len(df_entities)}")

print("\n=== Variants Found ===")
print(df_variants[["pmid", "variant", "normalized_hgvs", "clinical_significance"]].head(20))
print(f"\nTotal variants: {len(df_variants)}")

# Save 
df_entities.to_csv("results/paper_by_paper_entities.csv", index=False)
df_variants.to_csv("results/extracted_variants.csv", index=False)

print("\nResults saved to 'results/' directory")



Variants extracted and Normalizing via ClinVar...
  c.5946delT → N/A [N/A]
  c.5266dupC → N/A [N/A]
  c.68_69delAG → N/A [N/A]

=== Entities Found ===
   paper_id                                              title  \
0         0  Prevalence and Spectrum of BRCA Germline Varia...   
1         0  Prevalence and Spectrum of BRCA Germline Varia...   
2         0  Prevalence and Spectrum of BRCA Germline Varia...   
3         0  Prevalence and Spectrum of BRCA Germline Varia...   
4         0  Prevalence and Spectrum of BRCA Germline Varia...   

              text                 label       model  
0           breast                CANCER  BioNLP13CG  
1  ovarian cancers                CANCER  BioNLP13CG  
2          BRCA1/2  GENE_OR_GENE_PRODUCT  BioNLP13CG  
3           cancer                CANCER  BioNLP13CG  
4           breast                CANCER  BioNLP13CG  

Total entities: 101

=== Variants Found ===
       pmid       variant normalized_hgvs clinical_significance
0  31507061 

In [ ]:
# 7. Evaluation Metrics